## Final Project
# **RegTech Platform for Regulatory Surveillance**

#### **STEP 1. Data Ingestion and Extraction (Data Collection)**
- Explanation: Creation of the modules responsible for visiting the EMA and AEMPS websites to extract regulatory texts. It will be designed with an object-oriented approach, creating independent extractors for each agency that handle exceptions and network failures gracefully.
- Technologies: Python, scraping libraries like BeautifulSoup , and Requests.

In [1]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
from bs4 import BeautifulSoup
import time


def scrape_ema_multiple_baselines(url_list):
    """
    Iterates over a list of dictionaries containing URLs and target filenames.
    Scrapes each EMA page to establish multiple Scenario 1 (Baseline) files.
    Includes a polite delay between requests to prevent IP blocking.
    """
    # Define request headers to simulate a real browser session
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    counter=0
    for item in url_list:
        target_url = item.get("url")
        output_filename = item.get("filename")
        counter += 1
        try:
            print(f"Connecting to EMA target URL: {target_url}...")
            response = requests.get(target_url, headers=headers, timeout=15)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract the main document title
                title_element = soup.find('h1')
                title_text = title_element.get_text(strip=True) if title_element else "No Title Found"
                
                # Extract semantic text paragraphs (ignoring short UI/menu elements)
                paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 20]
                
                # Track links and attached documents to detect additions/deletions of PDFs
                tracked_links = []
                for anchor in soup.find_all('a', href=True):
                    anchor_text = anchor.get_text(strip=True)
                    href = anchor['href']
                    
                    # Filter for meaningful anchor texts or explicit document extensions
                    is_document = any(ext in href.lower() for ext in ['.pdf', '.doc', '.docx', '.xlsx'])
                    if len(anchor_text) > 10 or is_document:
                        tracked_links.append(f"LINK_TEXT: {anchor_text} | TARGET_URL: {href}")
                
                # Consolidate all extracted data into a single structured string
                consolidated_content = f"TITLE: {title_text}\n\n"
                consolidated_content += "=== TEXT CONTENT ===\n" + "\n\n".join(paragraphs) + "\n\n"
                consolidated_content += "=== ATTACHMENTS AND TRACKED LINKS ===\n" + "\n".join(tracked_links)
                
                # Persist data locally as the ground truth baseline
                with open(output_filename, "w", encoding="utf-8") as file:
                    file.write(consolidated_content)
                                       
                print(f"Success! Data saved to: '{output_filename}'\n")
                
            else:
                print(f"Access denied for {target_url}. HTTP Status Code: {response.status_code}\n")
                
        except Exception as error:
            print(f"An error occurred while processing {target_url}: {error}\n")
            
        # Polite scraping delay to avoid server overload or rate limiting
        print("Waiting 2 seconds before the next request...")
        time.sleep(2) 
        
    print(f"All URLs have been processed successfully. Total processed: {counter}")


In [11]:
# --- Execution Block ---
import os

if __name__ == "__main__":
    
    # 1. Folder & version
    output_dir = "Data_Extraction"
    version = "v1.2_"
    
    # 2. Create the folder at the same level as the notebook (exist_ok=True prevents errors if it exists)
    os.makedirs(output_dir, exist_ok=True)
    
    # 3. Define the URLs and build the file paths safely
    ema_target_sections = [
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview", "filename": os.path.join(output_dir, version + "ema_scenario_1_regulatory.txt")},
        {"url": "https://www.ema.europa.eu/en/news-events", "filename": os.path.join(output_dir, version + "ema_scenario_1_news.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_reg_research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/advanced-therapy-medicinal-products-overview", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_advanced-therapy-medicinal-products-overview.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/cancer-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_cancer-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/clinical-trials-human-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_clinical-trials-human-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-laboratory-practice-compliance", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-laboratory-practice-compliance.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-manufacturing-practice", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-manufacturing-practice.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/compliance-post-authorisation/good-distribution-practice", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_post-authorisation_compliance-post-authorisation_good-distribution-practice.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/quality-design", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_quality-design.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A50", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Compliance_inspections.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A56", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Regulatory_procedural_guidance.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A72", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Research_development.txt")},
        {"url": "https://www.ema.europa.eu/en/data-medicines-iso-idmp-standards-research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_data-medicines-iso-idmp-standards-research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/prime-priority-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_prime-priority-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/scientific-advice-protocol-assistance", "filename": os.path.join(output_dir, version + "ema_scenario_1_research-development_scientific-advice-protocol-assistance.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/marketing-authorisation", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_marketing-authorisation.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_post-authorisation.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/certification-medicinal-products", "filename": os.path.join(output_dir, version + "ema_scenario_1_post-authorisation_certification-medicinal-products.txt")},
        {"url": "https://health.ec.europa.eu/medicinal-products/eudralex_en", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_medicinal-products_eudralex_en.txt")}
    ] 
   
    # ==========================================
    # PHASE 1: DATA SCRAPING
    # ==========================================
    print("\n=== STARTING DATA EXTRACTION (SCRAPING) ===")
    # Call your intact scraping function
    scrape_ema_multiple_baselines(ema_target_sections)
    


=== STARTING DATA EXTRACTION (SCRAPING) ===
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_regulatory.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/news-events...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_news.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview/research-development...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_Hum_reg_research-development.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview/advanced-therapy-medicinal-products-overview...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_Hum_advanced-therapy-medicinal-products-overview.txt'

Waiting 2 seconds before the next request...
Connectin

#### **STEP 2. Historical Storage and Versioning (Database)**
- Explanation: Design of a centralized repository to store the raw extracted texts along with their metadata (agency, extraction date, link). This layer will ensure the temporal traceability needed to compare the "before" and "after" in on-demand queries.
- Technologies: A relational database approach was chosen, implemented via SQLite and managed through the SQLAlchemy ORM in Python

In [ ]:
import os

from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime


# ==========================================
# DATABASE SCHEMA SETUP
# ==========================================
Base = declarative_base()

class RegulatorySnapshot(Base):
    __tablename__ = 'regulatory_snapshots'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    agency = Column(String(50), nullable=False)
    url = Column(String(255), nullable=False)
    extraction_date = Column(DateTime, default=datetime.utcnow)
    title = Column(String(255))
    text_content = Column(Text, nullable=False)
    tracked_links = Column(Text)
    
class RegulatoryReport(Base):
    __tablename__ = 'regulatory_reports'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    execution_date = Column(DateTime, default=datetime.utcnow)
    total_alerts_captured = Column(Integer, nullable=False)
    english_summary = Column(Text, nullable=False)
    spanish_summary = Column(Text, nullable=False)
    raw_llm_payload = Column(Text, nullable=True)

os.makedirs('BD', exist_ok=True)
engine = create_engine('sqlite:///BD/regtech_data.db', echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

# ==========================================
# FUNCTIONS
# ==========================================
def save_snapshot_to_db(session, agency, url, title, text_content, tracked_links):
    """Saves a new scraped regulatory snapshot into the database."""
    try:
        new_snapshot = RegulatorySnapshot(
            agency=agency,
            url=url,
            title=title,
            text_content=text_content,
            tracked_links=tracked_links
        )
        session.add(new_snapshot)
        session.commit()
        return True
    except Exception as e:
        session.rollback()
        print(f"  [!] Failed to save to database: {e}")
        return False

def parse_scraped_file(filepath):
    """Reads the local .txt file and splits the Title, Content, and Links."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    title, text_content, tracked_links = "No Title", "", ""

    try:
        if "TITLE:" in content:
            title = content.split("=== TEXT CONTENT ===")[0].replace("TITLE:", "").strip()
        if "=== TEXT CONTENT ===" in content and "=== ATTACHMENTS AND TRACKED LINKS ===" in content:
            text_content = content.split("=== TEXT CONTENT ===")[1].split("=== ATTACHMENTS AND TRACKED LINKS ===")[0].strip()
        if "=== ATTACHMENTS AND TRACKED LINKS ===" in content:
            tracked_links = content.split("=== ATTACHMENTS AND TRACKED LINKS ===")[1].strip()
    except Exception as e:
        print(f"  [!] Error parsing structure in {filepath}: {e}")

    return title, text_content, tracked_links

# ==========================================
#  DATABASE LOADING
# ==========================================
print("\n=== STARTING SQLITE DATABASE LOAD ===")
    
db_session = Session()
uploaded_files = 0
failed_files = []
    
for item in ema_target_sections:
    filepath = item["filename"]
    target_url = item["url"]
    filename_only = os.path.basename(filepath)
        
    if os.path.exists(filepath):
        parsed_title, parsed_text, parsed_links = parse_scraped_file(filepath)
            
        # Syntax fixed: No line break before parentheses
        success = save_snapshot_to_db(
                session=db_session,
                agency="EMA",
                url=target_url,
                title=parsed_title,
                text_content=parsed_text,
                tracked_links=parsed_links
            )
            
        if success:
            uploaded_files += 1
        else:
            failed_files.append(f"{filename_only} (Failed to insert into DB)")
    else:
        failed_files.append(f"{filename_only} (File not found. Possible extraction failure.)")
            
db_session.close()
    
# ==========================================
# PHASE 3: FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("📋 FINAL PIPELINE REPORT")
print("="*50)
print(f"✅ Total URLs processed and saved to DB: {uploaded_files} out of {len(ema_target_sections)}")
    
if len(failed_files) > 0:
    print(f"❌ Files NOT processed ({len(failed_files)}):")
    for failure in failed_files:
        print(f"   - {failure}")
    else:
        print("🎉 End-to-End Pipeline completed without errors! All texts are in SQLite.")
print("="*50)


=== STARTING SQLITE DATABASE LOAD ===

📋 FINAL PIPELINE REPORT
✅ Total URLs processed and saved to DB: 21 out of 21


In [6]:
### ONLY DB SETUP - FOR DEBUGGING PURPOSES, THIS BLOCK CAN BE RUN INDEPENDENTLY TO SET UP THE DATABASE SCHEMA WITHOUT EXECUTING THE SCRAPING LOGIC.

import os
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime


# ==========================================
# DATABASE SCHEMA SETUP
# ==========================================
Base = declarative_base()

class RegulatorySnapshot(Base):
    __tablename__ = 'regulatory_snapshots'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    agency = Column(String(50), nullable=False)
    url = Column(String(255), nullable=False)
    extraction_date = Column(DateTime, default=datetime.utcnow)
    title = Column(String(255))
    text_content = Column(Text, nullable=False)
    tracked_links = Column(Text)
    
class RegulatoryReport(Base):
    __tablename__ = 'regulatory_reports'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    execution_date = Column(DateTime, default=datetime.utcnow)
    total_alerts_captured = Column(Integer, nullable=False)
    english_summary = Column(Text, nullable=False)
    spanish_summary = Column(Text, nullable=False)
    raw_llm_payload = Column(Text, nullable=True)

os.makedirs('BD', exist_ok=True)
engine = create_engine('sqlite:///BD/regtech_data.db', echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

#### **STEP 3. Change Detection Engine (NLP / Machine Learning)**
- Explanation: Instead of performing an exact text comparison (which would fail if a simple period or comma changes), paragraphs will be transformed into numerical vectors. By calculating the mathematical distance between last month's text and the current one, the model will detect if there has been a real change in meaning in the regulation, isolating only the important fragments.
- Technologies: Lightweight Transformer models (Sentence Transformers), libraries like HuggingFace, PyTorch, or Scikit-learn to calculate Cosine Similarity.

##### File nlp_change_detector.py created.
- It contains the implementation of the NLP-based change detection engine. It loads a pre-trained SentenceTransformer model to convert text into embeddings and calculates cosine similarity to identify significant changes in regulatory texts.

In [7]:
### ===============================================
### ======== COMPARE BASELINE VS ESCENARIO2 =======
### ===============================================

# Jupyter magic commands to automatically reload external modules
%load_ext autoreload
%autoreload 2

import os
# Disable the HuggingFace symlinks warning on Windows
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

from sqlalchemy import desc

# Import tools from your local module
from nlp_change_detector import hybrid_change_detector, extract_text_content

print("=== REGTECH AUTOMATED SURVEILLANCE PIPELINE ===")

# 1. Define the directory containing the "new" scraped data (Scenario 2)
scenario_2_dir = "Data_test_scenario_2_easy"
version = "v1.2_"

# 2. Master list of URLs (We use this to guide the loop and DB queries)
ema_target_sections = [
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview", "filename": version + "ema_scenario_1_regulatory.txt"},
    {"url": "https://www.ema.europa.eu/en/news-events", "filename": version + "ema_scenario_1_news.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development", "filename": version + "ema_scenario_1_Hum_reg_research-development.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/advanced-therapy-medicinal-products-overview", "filename": version + "ema_scenario_1_Hum_advanced-therapy-medicinal-products-overview.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/cancer-medicines", "filename": version + "ema_scenario_1_Hum_research-development_cancer-medicines.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/clinical-trials-human-medicines", "filename": version + "ema_scenario_1_Hum_research-development_clinical-trials-human-medicines.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development", "filename": version + "ema_scenario_1_Hum_research-development_compliance-research-development.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-laboratory-practice-compliance", "filename": version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-laboratory-practice-compliance.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-manufacturing-practice", "filename": version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-manufacturing-practice.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/compliance-post-authorisation/good-distribution-practice", "filename": version + "ema_scenario_1_Hum_post-authorisation_compliance-post-authorisation_good-distribution-practice.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/quality-design", "filename": version + "ema_scenario_1_Hum_research-development_quality-design.txt"},
    {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A50", "filename": version + "ema_scenario_1_Hum_Compliance_inspections.txt"},
    {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A56", "filename": version + "ema_scenario_1_Hum_Regulatory_procedural_guidance.txt"},
    {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A72", "filename": version + "ema_scenario_1_Hum_Research_development.txt"},
    {"url": "https://www.ema.europa.eu/en/data-medicines-iso-idmp-standards-research-development", "filename": version + "ema_scenario_1_Hum_data-medicines-iso-idmp-standards-research-development.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/prime-priority-medicines", "filename": version + "ema_scenario_1_Hum_research-development_prime-priority-medicines.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/scientific-advice-protocol-assistance", "filename": version + "ema_scenario_1_research-development_scientific-advice-protocol-assistance.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/marketing-authorisation", "filename": version + "ema_scenario_1_Hum_marketing-authorisation.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation", "filename": version + "ema_scenario_1_Hum_post-authorisation.txt"},
    {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/certification-medicinal-products", "filename": version + "ema_scenario_1_post-authorisation_certification-medicinal-products.txt"},
    {"url": "https://health.ec.europa.eu/medicinal-products/eudralex_en", "filename": version + "ema_scenario_1_Hum_medicinal-products_eudralex_en.txt"}
]

# 3. Open Database Session
db_session = Session()
total_alerts = 0
files_processed = 0
all_detected_changes = []
new_baselines_payload = {}    # Store the complete text associated with each URL

print("Connecting to the database to compare baseline vs new scraping...")

# 4. Main Processing Loop
for item in ema_target_sections:
    target_url = item["url"]
    
    # Define both possible filenames
    filename_scenario_1 = item["filename"]
    filename_scenario_2 = item["filename"].replace("scenario_1", "scenario_2")
    
    # Build full paths for the Scenario 2 directory
    path_option_1 = os.path.join(scenario_2_dir, filename_scenario_1)
    path_option_2 = os.path.join(scenario_2_dir, filename_scenario_2)
    
    # Intelligent check: Determine which file actually exists in the folder
    new_file_path = None
    if os.path.exists(path_option_2):
        new_file_path = path_option_2
    elif os.path.exists(path_option_1):
        new_file_path = path_option_1
        
    # If a valid file was found, proceed with the analysis
    if new_file_path:
        files_processed += 1
        
        # STEP A: Read the NEW text (Simulating the live website right now)
        # print("FILE REVIEWED-->", os.path.basename(new_file_path))
        new_regulatory_text = extract_text_content(new_file_path)
        # save the complete text in memory in case there are alerts and we need to update the DB
        new_baselines_payload[target_url] = new_regulatory_text
        
        # STEP B: Fetch the LAST KNOWN version from the Database
        last_snapshot = db_session.query(RegulatorySnapshot).filter_by(url=target_url).order_by(desc(RegulatorySnapshot.extraction_date)).first()
        
        if last_snapshot:
            old_regulatory_text = last_snapshot.text_content
            
            # STEP C: Compare using the Hybrid NLP Engine
            detected_changes = hybrid_change_detector(old_regulatory_text, new_regulatory_text, semantic_threshold=0.95)
                      
            if detected_changes:
                # Save URL and add it to the master list
                for change in detected_changes:
                 change['url'] = target_url

                all_detected_changes.extend(detected_changes)
                
                total_alerts += len(detected_changes)
                print(f"\n🚨 ALERT! Changes detected in URL: {target_url}")
                
                for idx, change in enumerate(detected_changes, 1):
                    print(f"   [Change #{idx}] - TYPE: {change['type']} (Similarity: {change['similarity']})")
                    if "MODIFIED" in change['type']:                       
                        print(f"   [-] DB VERSION: {change['old_text']}...")
                        
                    print(f"   [+] NEW VERSION: {change['new_text']}...")
                    print("   " + "-" * 100)
            else:
                pass # No significant changes detected, move to the next URL
                
        else:
            print(f"ℹ️ First time analyzing this URL (No baseline found in DB): {target_url}")
            
    else:
        pass # If the file exists under neither name, it simply skips it

# Close Database Session
db_session.close()

# 5. Final Report
print("\n" + "="*70)
print(f"🏁 SURVEILLANCE COMPLETE")
print(f"   - Files checked: {files_processed}")
print(f"   - Total critical alerts generated: {total_alerts}")
print("="*70)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
=== REGTECH AUTOMATED SURVEILLANCE PIPELINE ===
Connecting to the database to compare baseline vs new scraping...

🚨 ALERT! Changes detected in URL: https://www.ema.europa.eu/en/human-regulatory-overview
   [Change #1] - TYPE: MODIFIED (Critical Dates/Numbers) (Similarity: 0.9478)
   [-] DB VERSION: For further information on EU legislation and procedures for the regulation of human medicines, see volumes 1-4 and 9-10 of therules governing medicinal products in the EU....
   [+] NEW VERSION: For further information on EU legislation and procedures for the regulation of human medicines, see volumes 2-5 and 9-11 of therules governing medicinal products in the EU....
   ----------------------------------------------------------------------------------------------------
   [Change #2] - TYPE: MODIFIED (Semantic Change) (Similarity: 0.1513)
   [-] DB VERSION: Product emergency hotline...
   [+] NEW VERSI

In [5]:
# ==========================================
# UTILITY: DATABASE CONNECTION CLEANUP
# ==========================================
# Run this cell to safely release the SQLite .db file lock
# without needing to restart the Jupyter Kernel.

print("Initiating database teardown protocol...")

# 1. Close the main session from Step 3 (if it exists)
if 'db_session' in locals():
    try:
        db_session.close()
        print("✔ Main database session closed.")
    except Exception:
        pass

# 2. Close the consolidation session from Step 5 (if it exists)
if 'consolidation_session' in locals():
    try:
        consolidation_session.close()
        print("✔ Consolidation database session closed.")
    except Exception:
        pass

# 3. Dispose the engine to destroy the connection pool and release the file lock
if 'engine' in locals():
    try:
        engine.dispose()
        print("✔ Engine disposed. The '.db' file lock has been released.")
    except Exception:
        pass

print("✅ Environment clean. You can now safely delete or replace the database file.")

Initiating database teardown protocol...
✔ Main database session closed.
✔ Engine disposed. The '.db' file lock has been released.
✅ Environment clean. You can now safely delete or replace the database file.


In [30]:
## For debugging purposes, print all detected changes in a readable format
import json
####  indent=4 le da los espacios, y ensure_ascii=False respeta los acentos y caracteres especiales
print(json.dumps(all_detected_changes, indent=4, ensure_ascii=False))

[
    {
        "type": "MODIFIED (Critical Dates/Numbers)",
        "similarity": 0.9478,
        "old_text": "For further information on EU legislation and procedures for the regulation of human medicines, see volumes 1-4 and 9-10 of therules governing medicinal products in the EU.",
        "new_text": "For further information on EU legislation and procedures for the regulation of human medicines, see volumes 2-5 and 9-11 of therules governing medicinal products in the EU.",
        "url": "https://www.ema.europa.eu/en/human-regulatory-overview"
    },
    {
        "type": "MODIFIED (Semantic Change)",
        "similarity": 0.1513,
        "old_text": "Product emergency hotline",
        "new_text": "NEW ANNEX: Modified Prohibited Substances list",
        "url": "https://www.ema.europa.eu/en/human-regulatory-overview"
    },
    {
        "type": "ADDED",
        "similarity": 0.0,
        "old_text": null,
        "new_text": "This process requires companies to complete a new for

#### **STEP 4. Synthesis and Translation (Generative AI)**
- Explanation: Processing of the regulatory fragments isolated in the previous step. Structured system prompts will be designed so the LLM acts as a regulatory analyst, receives texts in English (EMA) or Spanish (AEMPS), and always returns a standardized executive summary in Spanish, organized in bullet points.
- Technologies: LLM APIs (Groq - Transformers - llama-3.1-8b-instant), orchestration frameworks like LangChain to handle call chains and context injection.

In [8]:
import os
import requests
import json
from dotenv import load_dotenv

# Load the environment variables from the .env file
load_dotenv()

def generate_bilingual_report_groq(detected_changes, groq_api_key):
    """
    Calls the Groq API (Llama-3.1-8B) to generate a dual-language regulatory report.
    Includes robust flexible parsing to split English and Spanish sections safely.
    """
    api_url = "https://api.groq.com/openai/v1/chat/completions"
    
    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }

    # Format the mathematically audited changes from Step 3 into context for the LLM
    changes_context = ""
    for idx, change in enumerate(detected_changes, 1):
        old_txt = change.get('old_text', '')
        new_txt = change.get('new_text', '')
        
        old_txt_cleaned = str(old_txt).replace('"', "'")[:600] if old_txt else ""
        new_txt_cleaned = str(new_txt).replace('"', "'")[:600] if new_txt else ""
        
        changes_context += f"Change #{idx} ({change['type']}) in URL: {change.get('url', 'Unknown')}\n"
        if old_txt_cleaned:
            changes_context += f"- Previous text in DB: {old_txt_cleaned}...\n"
        changes_context += f"- New text scraped: {new_txt_cleaned}...\n"
        changes_context += "-" * 40 + "\n"

    # Define the strict rules and context using the chat messages format
    payload = {
        "model": "llama-3.1-8b-instant", 
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a senior Pharmaceutical Regulatory Consultant. Your job is to analyze technical "
                    "changes in EMA guidelines.\n"
                    "You must provide the executive summary in TWO distinct sections: English and Spanish.\n\n"
                    "STRICT RULES:\n"
                    "1. Summarize only the real regulatory impact of the changes.\n"
                    "2. For each change, structure the output as follows: row 1 - URL, row 2 - type of change, row 3 - the summary of the change.\n"
                    "3. Use a highly professional, formal, and corporate tone.\n"
                    "4. Format the response using clear bullet points for each language and include appropriate icons.\n"
                    "5. Never alter or translate the domain or path of the URLs provided. Keep them exactly as they appear in the source data.\n"
                    "6. Strictly adhere to this structural output layout:\n"
                    "[ENGLISH_VERSION]\n"
                    "(Your English bullet points summarizing the changes here)\n\n"
                    "[SPANISH_VERSION]\n"
                    "(Your Spanish bullet points summarizing the changes here)"
                )
            },
            {
                "role": "user",
                "content": f"Here are the detected changes to analyze:\n\n{changes_context}"
            }
        ],
        "temperature": 0.1,  
        "max_tokens": 3000
    }

    try:
        response = requests.post(api_url, headers=headers, json=payload)
        
        if response.status_code != 200:
            print(f"Groq API Error Details: {response.text}")
        response.raise_for_status()
        
        result_json = response.json()
        raw_result = result_json['choices'][0]['message']['content'].strip()
        
        # --- ROBUST BILINGUAL PARSING LAYER ---
        english_part = raw_result
        spanish_part = ""
        
        # We look for multiple layout variations that Llama might generate for the Spanish marker
        spanish_markers = ["[SPANISH_VERSION]", "--- SPANISH VERSION ---", "**SPANISH VERSION**", "SPANISH VERSION:"]
        
        for marker in spanish_markers:
            if marker in raw_result:
                parts = raw_result.split(marker)
                english_part = parts[0]
                # In case the marker repeats at the end, we grab the immediate content block
                spanish_part = parts[1].split("--- SPANISH VERSION ---")[0].strip() if len(parts) > 1 else parts[1]
                break
        
        # Clean up tags from the English block as well
        for marker in ["[ENGLISH_VERSION]", "--- ENGLISH VERSION ---", "**ENGLISH_VERSION**"]:
            english_part = english_part.replace(marker, "")
            
        report_data = {
            "full_text": raw_result,
            "en": english_part.strip(),
            "es": spanish_part.strip() if spanish_part else "Error: Could not isolate Spanish version dynamically."
        }
        return report_data

    except Exception as e:
        print(f"Error calling Groq Llama-3 API: {e}")
        return None

# ==========================================
# EXECUTION BLOCK (STEP 4)
# ==========================================
if __name__ == "__main__":
    
    # 1. Fetch the Groq API key safely from the environment
    GROQ_KEY = os.getenv("GROQ_API_KEY")

    if not GROQ_KEY:
        print("Error: GROQ_API_KEY not found. Please verify your .env file.")
    else:
        # 2. Link with Step 3. Fallback to mock data if Step 3 variable is missing.
        try:
            # We now read from the global accumulator list created in Step 3
            changes_to_process = all_detected_changes 
            print(f"Successfully loaded {len(changes_to_process)} changes from the Step 3 pipeline.")
        except NameError:
            print("Step 3 variable 'all_detected_changes' not found. Loading Mock Data for testing...")
            changes_to_process = [
                {
                    "type": "MODIFIED (Critical Dates/Numbers)",
                    "url": "https://www.ema.europa.eu/en/human-regulatory-overview",
                    "old_text": "The document was open for consultation from 4 March to 30 April 2026.",
                    "new_text": "The document was open for consultation from 4 January to 30 April 2026."
                }
            ]

        # 3. Trigger the generation using the internal traceability naming convention
        if len(changes_to_process) > 0:
            print("Contacting Groq API cloud infrastructure...")
            lastdance_report = generate_bilingual_report_groq(changes_to_process, GROQ_KEY)

            if lastdance_report:
                print("\n" + "="*60)
                print("📊 LAST DANCE BILINGUAL REPORT GENERATED (via GROQ)")
                print("="*60)
                
                print("\n--- ENGLISH VERSION ---")
                print(lastdance_report['en'])
                
                print("\n--- SPANISH VERSION ---")
                print(lastdance_report['es'])
                print("="*60)
        else:
            print("No relevant modifications detected in Step 3 to build a report.")

Successfully loaded 12 changes from the Step 3 pipeline.
Contacting Groq API cloud infrastructure...

📊 LAST DANCE BILINGUAL REPORT GENERATED (via GROQ)

--- ENGLISH VERSION ---
**ENGLISH VERSION**

[Changes to EMA Guidelines]

• **Change #1:** https://www.ema.europa.eu/en/human-regulatory-overview
  • Type of change: MODIFIED (Critical Dates/Numbers)
  • Summary of change: The volumes of the rules governing medicinal products in the EU have been updated from 1-4 and 9-10 to 2-5 and 9-11.

• **Change #2:** https://www.ema.europa.eu/en/human-regulatory-overview
  • Type of change: MODIFIED (Semantic Change)
  • Summary of change: The term "Product emergency hotline" has been replaced with "NEW ANNEX: Modified Prohibited Substances list".

• **Change #3:** https://www.ema.europa.eu/en/human-regulatory-overview
  • Type of change: ADDED
  • Summary of change: A new process has been introduced requiring companies to complete a new form for critical drug traceability tracking.

• **Change #

In [32]:

columnas = list(changes_to_process[0].keys())
print(columnas)

['type', 'similarity', 'old_text', 'new_text', 'url']


#### **5. Backend Development and Business Logic (API)**
- Explanation: Construction of the core engine that connects the database, NLP models, and the LLM. This is where the endpoints that will execute on-demand reports and manage the CRUD (Create, Read, Update, Delete) operations of the user database are defined. Solid data validation and unit testing strategies will be applied to guarantee service stability.
- Technologies: FastAPI (for its extremely high performance and automatic documentation), Pydantic (data schema validation), Pytest (for test automation).

In [9]:
import os
import datetime
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

# --- Assuming these are your existing SQLAlchemy model definitions ---
# don't need to re-declare them, but they are placed here as a reference of what we are updating

# The 'RegulatorySnapshot' class and 'Session' factory are already defined and imported.

def consolidate_surveillance_cycle(db_session, processed_changes, complete_texts, bilingual_report):
    """
    Consolidates the surveillance run by updating the database baseline 
    with the full new text, and permanently logging the generated AI executive report.
    """
    print("Starting database consolidation and baseline synchronization...")
    
    try:
        # 1. Update Baseline Snapshots
        unique_urls = set([change.get('url') for change in processed_changes if change.get('url')])
        for target_url in unique_urls:
            snapshot = db_session.query(RegulatorySnapshot).filter_by(url=target_url).first()
            if snapshot:
                snapshot.text_content = complete_texts[target_url]
                snapshot.extraction_date = datetime.datetime.now()
                print(f"✔ Baseline successfully updated for: {target_url}")
            else:
                print(f"⚠ Warning: Snapshot not found in DB for URL: {target_url}")

        # 2. LOG THE AI REPORT INTO THE DATABASE (REAL INSERT)
        new_report_entry = RegulatoryReport(
            execution_date=datetime.datetime.now(),
            total_alerts_captured=len(processed_changes),
            english_summary=bilingual_report.get('en'),
            spanish_summary=bilingual_report.get('es'),
            raw_llm_payload=bilingual_report.get('full_text')
        )
        
        db_session.add(new_report_entry)
        
        # 3. Commit the transaction
        db_session.commit()
        print("\n✔ Historical audit trail saved. Transaction committed successfully.")
        return True

    except Exception as e:
        db_session.rollback()
        print(f"\n❌ Critical error during database consolidation: {e}")
        print("Database transaction rolled back safely.")
        return False
    
# ==========================================
# EXECUTION BLOCK (STEP 5)
# ==========================================
if __name__ == "__main__":
    
    print("=== REGTECH PHASE 5: DATABASE CONSOLIDATION ===")
    
    pipeline_ready = True
    
    # 1. Safety verification: Ensure data from Step 3 and Step 4 exists in memory
    try:
        changes_payload = all_detected_changes
        texts_payload = new_baselines_payload
    except NameError:
        print("Error: Variables from Step 3 not found. Please run the Step 3 cell first.")
        pipeline_ready = False
        
    try:
        report_payload = lastdance_report
        if report_payload is None:
            pipeline_ready = False
            print("Error: The AI report from Step 4 is empty or failed.")
    except NameError:
        pipeline_ready = False
        print("Error: 'lastdance_report' from Step 4 was not found in the environment.")

    # 2. Run consolidation if the pipeline is healthy
    if pipeline_ready:
        
        # We instantiate a new secure session for the consolidation process
        try:
            consolidation_session = Session()
            
            success = consolidate_surveillance_cycle(
                consolidation_session, 
                changes_payload, 
                texts_payload, 
                report_payload
            )
            
            consolidation_session.close()
            
            if success:
                print("\n" + "="*60)
                print("🏁 PIPELINE CYCLE FULLY COMPLETED AND CLOSED")
                print("All systems synchronized. The database is ready for the next Scenario.")
                print("="*60)
                
        except Exception as db_error:
            print(f"Failed to open database session: {db_error}")
    else:
        print("\nConsolidation aborted due to missing upstream pipeline data.")

=== REGTECH PHASE 5: DATABASE CONSOLIDATION ===
Starting database consolidation and baseline synchronization...
✔ Baseline successfully updated for: https://www.ema.europa.eu/en/human-regulatory-overview/research-development/clinical-trials-human-medicines
✔ Baseline successfully updated for: https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A50
✔ Baseline successfully updated for: https://www.ema.europa.eu/en/human-regulatory-overview

✔ Historical audit trail saved. Transaction committed successfully.

🏁 PIPELINE CYCLE FULLY COMPLETED AND CLOSED
All systems synchronized. The database is ready for the next Scenario.


#### **6. Management Interface and Frontend (UI)**
- Explanation: Creation of the visual control panel. It will have two main sections: a manager to add or remove emails from the distribution groups, and an analytical dashboard with date and agency selectors so a user can instantly generate custom regulatory reports.
- Technologies: Streamlit or Gradio (they allow building professional data interfaces directly in Python with great agility).

In [36]:
%%writefile app.py          
import datetime
import json
import os
import streamlit as st

#### %%writefile must be the first line in the cell to work properly in Jupyter. This command creates a new file.
##########################################################
########### CREATION OF THE STREAMLIT DASHBOARD ##########
##########################################################
### app.py - This is the Streamlit application that serves as the executive dashboard for the RegTech surveillance system. 
### It displays the AI-generated bilingual report and key metrics from the latest surveillance run.


# --- PAGE CONFIGURATION ---
st.set_page_config(
    page_title="RegTech AI Surveillance",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- HEADER ---
st.title("🛡️ RegTech Automated Surveillance System")
st.markdown("### Executive Regulatory Intelligence Dashboard")
st.markdown("---")

# --- SIMULATED DATA INGESTION ---
# In production, this would fetch from your SQLAlchemy 'RegulatoryReport' table
# For this MVP frontend, we load the JSON payload directly.
@st.cache_data
def load_mock_data():
    return {
        "execution_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_alerts": 12,
        "english_summary": """
• **Change #1 (MODIFIED)** in URL: https://www.ema.europa.eu/en/human-regulatory-overview
  • Summary: The volumes of the rules governing medicinal products in the EU have been updated from 1-4 and 9-10 to 2-5 and 9-11.
• **Change #4 (MODIFIED)** in URL: https://www.ema.europa.eu/en/human-regulatory-overview/clinical-trials
  • Summary: The initial advice for sponsors on clinical trials was issued on 15 March 2025.
        """,
        "spanish_summary": """
• **Cambio #1 (MODIFICADO)** en URL: https://www.ema.europa.eu/es/human-regulatory-overview
  • Resumen: Los volúmenes de las reglas que rigen los productos medicinales en la UE han sido actualizados a 2-5 y 9-11.
• **Cambio #4 (MODIFICADO)** en URL: https://www.ema.europa.eu/es/human-regulatory-overview/clinical-trials
  • Resumen: El consejo inicial para los patrocinadores sobre ensayos clínicos se emitió el 15 de marzo de 2025.
        """
    }

report_data = load_mock_data()

# --- SIDEBAR CONTROLS ---
with st.sidebar:
    st.header("⚙️ Dashboard Controls")
    language = st.radio("Select Report Language / Idioma:", ("English", "Español"))
    st.markdown("---")
    st.metric(label="Target Source", value="EMA Guidelines")
    st.metric(label="Last Execution", value=report_data["execution_date"])
    st.metric(label="Critical Alerts Detected", value=report_data["total_alerts"])
    
# --- MAIN DASHBOARD AREA ---
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader(f"📑 AI Executive Summary ({language})")
    st.info("Report generated by Groq Cloud (Llama 3.1) based on hybrid NLP diffs.")
    
    if language == "English":
        st.markdown(report_data["english_summary"])
    else:
        st.markdown(report_data["spanish_summary"])

with col2:
    st.subheader("📊 System Status")
    st.success("✅ Baseline Synchronized")
    st.success("✅ NLP Engine Online")
    st.success("✅ Database Committed")
    
    st.markdown("### Next Steps")
    st.button("Run Scenario 3 Verification", type="primary", use_container_width=True)

st.markdown("---")
st.caption("Developed by Juan Álvaro Romero | Ironhack Big Data & Machine Learning MVP | © 2026")

Writing app.py


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
### This block is for generating a Windows icon (.ico) from .png image. It uses the Pillow library to handle image processing.
# %pip install Pillow
from PIL import Image

# 1. Abrir la imagen generada por IA
img = Image.open('logo_regtech2.png')

# 2. Recortarla a un cuadrado perfecto (si no lo es) y redimensionar
# Los tamaños estándar para iconos de Windows son 256x256, 128x128, 64x64, 32x32
icon_sizes = [(256, 256), (128, 128), (64, 64), (32, 32)]

# 3. Guardarla directamente como archivo .ico
img.save('regtech_icon2.ico', format='ICO', sizes=icon_sizes)

print("¡Icono regtech_icon2.ico generado con éxito! Ya puedes usarlo en tu acceso directo.")

¡Icono regtech_icon.ico generado con éxito! Ya puedes usarlo en tu acceso directo.


In [10]:
%pip install markdown

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
